# Detector comparison on a GPU runtime (RT-DETR-l · YOLOv8s · YOLOv8n × base / augmented data)

Runtime → Change runtime type → **T4 GPU** (L4 also fine), then Runtime → Run all.

**Disconnect-safe.** Weights, logs, cached detections and results are written to Google Drive (`MyDrive/sss_bench`).
If the runtime disconnects (phone in background, timeout), reconnect and *Run all* again:
finished runs are skipped, interrupted runs resume from their last epoch. Only the data preparation (~10 min) is repeated.

Every run uses the same tiles, hyper-parameters, epochs and test-year evaluation (`src/b5_eval.py`):
point FROC on 578 test frames (2010 · 2018), Pd at 0.25–8 false alarms per frame, frame-bootstrap 95% CI,
full-frame (1024 px) and sliced (320 px tiles) inference. The final-epoch weights (`last.pt`) are evaluated,
because the 26-frame validation set is too noisy to pick a best epoch.

Runs are ordered by priority: RT-DETR first (it cannot be trained on the CPU used for the repository results).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/sss_bench"
import os; os.makedirs(DRIVE, exist_ok=True)

In [ ]:
%cd /content
!rm -rf sss-mine-detection-benchmark
!git clone -q https://github.com/jongchan-joseph-lee/sss-mine-detection-benchmark.git
%cd sss-mine-detection-benchmark/src
!pip -q install "ultralytics>=8.3" scipy scikit-learn
!mkdir -p {DRIVE}/b5_dets && rm -rf b5_dets && ln -s {DRIVE}/b5_dets b5_dets

## 1. Data and tiles (official year split: train 2015 · 2017 · 2021, test 2010 · 2018)

In [ ]:
!bash download_data.sh > /dev/null
!python prep_tiles.py      # base tiles, val split, 1024-px test frames
!python prep_aug.py        # scale-down + shadow-aware copy-paste + speckle/JPEG (training scenes only)

## 2. Training grid (skips finished runs, resumes interrupted ones)

In [ ]:
EPOCHS = 60
GRID = [("rtdetr", "rtdetr-l.pt", "aug"), ("rtdetr", "rtdetr-l.pt", "base"),
        ("s", "yolov8s.pt", "aug"), ("s", "yolov8s.pt", "base"),
        ("n", "yolov8n.pt", "aug"), ("n", "yolov8n.pt", "base")]
DATA = {"base": "tiles/milco_tiles.yaml", "aug": "tiles/milco_tiles_aug.yaml"}
runs = {}
for mk, w, dk in GRID:
    name = f"{mk}_{dk}{EPOCHS}"
    !python train_exp.py --auto --data {DATA[dk]} --model {w} --epochs {EPOCHS} --name {name} --device 0 --workers 4 --project {DRIVE}/runs
    runs[name] = f"{DRIVE}/runs/{name}/weights/last.pt"
runs

## 3. Test-year evaluation (reference = the CPU-trained YOLOv8n in the repository)

In [ ]:
args = " ".join(["ref=runs/detect/runs/yolo_tiles/weights/best.pt"] + [f"{k}={v}" for k, v in runs.items() if os.path.exists(v)])
!B5_DEVICE=0 B5_OUT=b5_eval_gpu.json python b5_eval.py {args}
!cp b5_eval_gpu.json {DRIVE}/b5_eval_gpu.json

In [ ]:
import json, pandas as pd
r = json.load(open("b5_eval_gpu.json"))
rows = []
for k, v in r.items():
    if "fa_grid" not in v:
        continue
    g = v["fa_grid"]; i1, i4 = g.index(1), g.index(4)
    d = v.get("diff_vs_ref_full", {})
    rows.append(dict(run=k, pd_at_1=v["pd"][i1], ci_1=v["ci95"][i1], pd_at_4=v["pd"][i4], ci_4=v["ci95"][i4],
                     pd_30_50px_at_4=v["pd_by_size_at_4fa"]["30-50"]["pd"],
                     diff_vs_ref_at_4=d.get("mean", [None] * len(g))[i4], diff_ci_at_4=d.get("ci95", [None] * len(g))[i4]))
pd.DataFrame(rows).set_index("run").round(3)

## 4. Results file (also kept in Google Drive → `MyDrive/sss_bench`)

In [ ]:
!cd {DRIVE} && zip -q -r /content/b5_colab_results.zip b5_eval_gpu.json b5_dets runs/*/weights/last.pt runs/*/results.csv
!cp /content/b5_colab_results.zip {DRIVE}/
from google.colab import files
files.download("/content/b5_colab_results.zip")